## Cicli innestati in OpenMP

- Nei cicli innestati, in OpenMP si parallelizza spesso solo il ciclo piu esterno con `#pragma omp parallel for`.
- Le iterazioni esterne vengono distribuite tra i thread; quelle interne restano sequenziali per ogni thread.
- Con carico squilibrato o molte iterazioni interne, conviene usare il collasso dei cicli.

### Collasso dei cicli

Con due cicli annidati di dimensioni `n` e `m`, OpenMP puo unirli in uno spazio di iterazione di dimensione `n * m` con:

`#pragma omp for collapse(2)`

In [5]:
#if defined(_OPENMP)
  #include <omp.h>
#endif
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[]) {
    const int n = 100000;
    int *v = (int *)malloc(n * sizeof(int));
    if (v == NULL) {
        fprintf(stderr, "malloc fallita\n");
        return 1;
    }

    // Questa dipendenza (v[i-10]) rende il ciclo non parallelizzabile in sicurezza.
    for (int i = 0; i < n; i++) {
        if (i > 10) {
            v[i] = i + v[i - 10];
        } else {
            v[i] = i;
        }
    }

    int sum = 0;
#pragma omp parallel for reduction(+ : sum)
    for (int i = 0; i < n; i++) {
        sum += v[i];
    }

    printf("sum = %d\n", sum);
    free(v);
    return 0;
}

sum = 148649224


## Task in OpenMP (`#pragma omp task`)

Usa i task quando il lavoro non e un semplice ciclo regolare, ma un insieme dinamico di sotto-problemi.

- Un thread crea i task (dispatcher).
- I thread del team li eseguono in modo asincrono.
- Utile per ricorsione (es. divide-et-impera), pipeline e workload irregolari.

## Chiamare C da Python con `ctypes`

Punti chiave:

- Definisci tipi di argomento (`argtypes`) e tipo di ritorno (`restype`).
- Gestisci con attenzione memoria e puntatori.
- Le stringhe Python sono Unicode, in C spesso `char*` (byte).

### Compilazione libreria condivisa

`gcc -fPIC -shared -o library.so library.c`

### Esempio puntatori (`swap`)

Codice C:

```c
#include <stdio.h>

void swap(int *a, int *b) {
    int tmp = *a;
    *a = *b;
    *b = tmp;
}
```

Codice Python:

```python
import ctypes

code = ctypes.cdll.LoadLibrary("./library.so")
code.swap.argtypes = [ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int)]
code.swap.restype = None

a = ctypes.c_int(4)
b = ctypes.c_int(5)

print(f"a = {a.value}, b = {b.value}")
code.swap(ctypes.pointer(a), ctypes.pointer(b))
print(f"a = {a.value}, b = {b.value}")
```

### Nota rapida sui puntatori

In Python, con `ctypes`, puoi ottenere il puntatore a una variabile con `ctypes.pointer(...)`:

```python
a = ctypes.c_int(4)
pa = ctypes.pointer(a)  # tipo: POINTER(c_int)
```

Quando una funzione C si aspetta `int*`, in `argtypes` devi usare `ctypes.POINTER(ctypes.c_int)`.